In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('train.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
num_df = df.select_dtypes(include='number')

In [ ]:
df['Irrigation_Need'] = df['Irrigation_Need'].map({'Low': 0,
                                                  'Medium': 1,
                                                  'High': 2})

In [ ]:
df['Mulching_Used'] = df['Mulching_Used'].map({'No': 0,
                                              'Yes':1}).astype(int)

In [ ]:
df['stage_irrigation'] = df['Crop_Growth_Stage'] + "_" + df['Irrigation_Type']

In [ ]:
cols = ['Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Region']
for col in cols:
    df[f'Soil_{col}'] = df['Soil_Type'] + "_" + df[col]

In [ ]:
cat_df = df.select_dtypes(exclude='number')

In [ ]:
interactions = [
    ('Crop_Type', 'Crop_Growth_Stage'),
    ('Crop_Growth_Stage', 'Season'),
    ('Season', 'Irrigation_Type'),
    ('Irrigation_Type', 'Water_Source'),
    ('Crop_Type', 'Season'),
    ('Crop_Type', 'Irrigation_Type'),
    ('Water_Source', 'Crop_Type'),
    ('Crop_Growth_Stage', 'Water_Source'),
    ('Season', 'Water_Source'),
    ('Region', 'Crop_Type'),
    ('Region', 'Crop_Growth_Stage'),
    ('Region', 'Season'),
    ('Region', 'Irrigation_Type'),
    ('Region', 'Water_Source'),
]

for a, b in interactions:
    df[f'{a}_{b}'] = df[a] + "_" + df[b]
# then target encode all

In [ ]:
X = df.drop(columns=['Irrigation_Need'])
y = df['Irrigation_Need']

In [ ]:
import category_encoders as ce
from sklearn.model_selection import train_test_split
import lightgbm as lgb

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

target_encode_cols = [
    'Crop_Type',
    'stage_irrigation', 
    'Soil_Crop_Type', 'Soil_Crop_Growth_Stage',
    'Soil_Season', 'Soil_Irrigation_Type', 'Soil_Region',
    'Crop_Type_Crop_Growth_Stage',
    'Crop_Growth_Stage_Season', 'Season_Irrigation_Type',
    'Irrigation_Type_Water_Source', 'Crop_Type_Season',
    'Crop_Type_Irrigation_Type', 'Water_Source_Crop_Type',
    'Crop_Growth_Stage_Water_Source', 'Season_Water_Source', 'Region_Water_Source', 
    'Region_Crop_Type', 'Region_Season', 'Region_Crop_Growth_Stage',
    'Region_Irrigation_Type'
]

encoder = ce.TargetEncoder(cols=target_encode_cols)
X_train[target_encode_cols] = encoder.fit_transform(X_train[target_encode_cols], y_train)
X_test[target_encode_cols] = encoder.transform(X_test[target_encode_cols])

ohe_cols = ['Soil_Type', 'Crop_Growth_Stage', 'Season', 
            'Irrigation_Type', 'Water_Source', 'Region']
X_train = pd.get_dummies(X_train, columns=ohe_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=ohe_cols, drop_first=True)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

train_columns = X_train.columns.tolist()

In [ ]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=1000, class_weight='balanced')

lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print(classification_report(y_test, y_pred))

In [ ]:
model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    class_weight='balanced',
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:
# TEST
test = pd.read_csv('test.csv')

# save id
test_ids = test['id']
test = test.drop(columns=['id'])

test['Mulching_Used'] = test['Mulching_Used'].map({'No': 0, 'Yes': 1}).astype(int)

# FEATURE ENGINEERING - same as train
test['stage_irrigation'] = test['Crop_Growth_Stage'] + "_" + test['Irrigation_Type']

cols = ['Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Region']
for col in cols:
    test[f'Soil_{col}'] = test['Soil_Type'] + "_" + test[col]

interactions = [
('Crop_Type', 'Crop_Growth_Stage'),
    ('Crop_Growth_Stage', 'Season'),
    ('Season', 'Irrigation_Type'),
    ('Irrigation_Type', 'Water_Source'),
    ('Crop_Type', 'Season'),
    ('Crop_Type', 'Irrigation_Type'),
    ('Water_Source', 'Crop_Type'),
    ('Crop_Growth_Stage', 'Water_Source'),
    ('Season', 'Water_Source'),
    ('Region', 'Crop_Type'),
    ('Region', 'Crop_Growth_Stage'),
    ('Region', 'Season'),
    ('Region', 'Irrigation_Type'),
    ('Region', 'Water_Source'),
]
for a, b in interactions:
    test[f'{a}_{b}'] = test[a] + "_" + test[b]

# TARGET ENCODE
test[target_encode_cols] = encoder.transform(test[target_encode_cols])

# OHE
test = pd.get_dummies(test, columns=ohe_cols, drop_first=True)

# ALIGN columns with train
test = test.reindex(columns=train_columns, fill_value=0)

# PREDICT
y_pred = model.predict(test)

# MAP back to labels
label_map = {0: 'Low', 1: 'Medium', 2: 'High'}
y_pred_labels = [label_map[p] for p in y_pred]

# SUBMISSION
submission = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': y_pred_labels
})
submission.to_csv('submission.csv', index=False)
print(submission.head())
print(submission['Irrigation_Need'].value_counts())